# Testing whether `rechp.maxeval = 2000` is actually needed

`rechp.xtol_rel = rechp.ftol_rel = rechp.ftol_abs = 0.0` in `scripts/endtoend_opt.jl`'s production
config, so the inner reconstruction (`reconstruct_Tmap_and_zmap_distributed`) currently never stops
early for any reason other than hitting `maxeval` -- confirmed by `MAXEVAL_REACHED` showing up on
nearly every outer iteration of the real run (`optdata/opt_2026-11-13T13:07:16.666/history.csv`),
which is a large part of why each outer iteration costs ~30 minutes.

This notebook instead enables `xtol_rel` (see the planning discussion for why `xtol_rel` and not
`ftol_rel`/`ftol_abs` -- the latter have a documented history of causing spurious premature
`:SUCCESS` in this exact pipeline, see the comment above `reconstruct_Tmap_and_zmap_distributed`,
`src/forward_distributed.jl`) at a sweep of values, and checks whether the reconstruction error
(`L_reconstruction_error`) actually changes once some looser tolerance stops it well short of 2000
iterations -- for both the run's "initial" geoms/α/β (pre-training) and its "checkpoint" geoms/α/β
(iteration 150, the last checkpoint before the job hit SLURM's wall-clock limit).

In [ ]:
cd(isdir("../../End2EndThermalImg.jl") ? "../../End2EndThermalImg.jl" : "../../End2EndThermalImg")
import Pkg
Pkg.activate(".")

In [ ]:
using Revise
using End2EndThermalImg
using PythonPlot
using Distributed
using LinearAlgebra
using Statistics
using DataFrames

## Setup

In [ ]:
@time nw = End2EndThermalImg.setup_cluster_workers()
println("nworkers = ", nworkers(), "  workers = ", workers())
println("threads per worker = ", [remotecall_fetch(Threads.nthreads, p) for p in workers()])

## Load the initial and checkpoint states

Both the "initial" (pre-training) and "checkpoint" (iteration 150) geoms/α/β come from the same
run's `results.jld2`, via `load_checkpoint_state` (`src/optimize.jl`, originally written for
`scripts/checkpoint_report.jl`). `history[1].α`/`.β` are exactly the resolved initial α/β the real
run's own `reconstruction_initial.png` used (the values passed into `train_outer_optimization_distributed`
before any Adam updates).

In [ ]:
run_folder = "optdata/opt_2026-11-13T13:07:16.666"  # the timed-out production job

state = End2EndThermalImg.load_checkpoint_state(run_folder)
php, imghp, opthp, rechp = state.php, state.imghp, state.opthp, state.rechp
PSF_scale = state.PSF_scale
geoms_init_flat, geoms_opt_flat = state.geoms_init_flat, state.geoms_opt_flat
α_init, β_init = state.history[1].α, state.history[1].β   # resolved initial α/β, pre-training
α_opt, β_opt = state.α_opt, state.β_opt                    # α/β as of the checkpoint
checkpoint_iteration = state.history[end].iteration

jhp = JobHyperParams(php = php, imghp = imghp, opthp = opthp, rechp = rechp)

println("checkpoint iteration = ", checkpoint_iteration)
println("α_init = ", α_init, "  β_init = ", β_init)
println("α_opt  = ", α_opt,  "  β_opt  = ", β_opt)

In [ ]:
# php/imghp/opthp/rechp are loaded at runtime on the driver only, so they need to be sent to every
# worker explicitly -- recompute_fftPSFs_shard! (src/forward_distributed.jl) reads Main.php
# resident on each worker (and Main.surrogates, set below, itself depends on php). Mirrors
# scripts/checkpoint_report.jl. rechp/opthp aren't actually read resident anywhere, but broadcasting
# all four keeps this identical to that script rather than relying on that distinction holding.
for (name, value) in ((:php, php), (:imghp, imghp), (:opthp, opthp), (:rechp, rechp))
    End2EndThermalImg.broadcast_value(name, value)
end

## Physics / FFT setup

In [ ]:
@time driver_plans_n2f, driver_plans_PSF = End2EndThermalImg.setup_fft_plans_distributed!(php, imghp)

In [ ]:
@time freqs = End2EndThermalImg.get_freq_chebpoints(php)
@time n2f_kernels = End2EndThermalImg.compute_and_broadcast(:n2f_kernels, End2EndThermalImg.get_n2f_kernels, freqs, driver_plans_n2f, php, imghp);

In [ ]:
@time @everywhere begin
    surrogates = End2EndThermalImg.load_surrogate_models(php)
    PSF_zcoords = End2EndThermalImg.get_PSF_zcoords(imghp)
    weights = End2EndThermalImg.get_clenshaw_curtis_quadrature_weights(php)
end

In [ ]:
@time End2EndThermalImg.assign_shard_indices!()
@time End2EndThermalImg.init_resident_incidents!(freqs, PSF_zcoords, php)

## Draw one random test object/noise

A plain (unseeded) draw, not the deterministic `with_tuning_rng` one used for
`reconstruction_initial*.png`. Reused as-is for every comparison below (both geoms states, every
`xtol_rel` value), so `xtol_rel` (and separately, geoms state) is the only thing actually varying.

In [ ]:
object = End2EndThermalImg.get_object(imghp)
noise = End2EndThermalImg.generate_noise(imghp)

## Helper: clone `rechp` with a different `xtol_rel`

`ReconstructionHyperParams` (`src/prepare.jl`) is a plain struct with a hand-written keyword
constructor, not `@with_kw`, so there's no built-in "clone with override" syntax here (unlike
`JobHyperParams` below, which is `@with_kw`) -- this just rebuilds it, passing every field through
unchanged except `xtol_rel`/`maxeval`.

In [ ]:
with_xtol_rel(rechp, xtol_rel; maxeval = rechp.maxeval) = ReconstructionHyperParams(
    object_init_type = rechp.object_init_type,
    T_background = rechp.T_background,
    z_middle_μm = rechp.z_middle_μm,
    algorithm = rechp.algorithm,
    α_init = rechp.α_init,
    β_init = rechp.β_init,
    maxeval = maxeval,
    xtol_rel = xtol_rel,
    ftol_rel = rechp.ftol_rel,
    ftol_abs = rechp.ftol_abs,
    php = php
)

## States and tolerances to sweep

`ftol_rel`/`ftol_abs` stay at `0.0` throughout -- only `xtol_rel` is being tested. `maxeval` is kept
fixed at a generous ceiling (not the thing driving convergence here); if some `xtol_rel` value still
hits it (`MAXEVAL_REACHED` instead of a tolerance-triggered return), that value was too tight to
matter at this budget.

In [ ]:
states = [
    ("initial",    geoms_init_flat, α_init, β_init),
    ("checkpoint", geoms_opt_flat,  α_opt,  β_opt),
]

xtol_rels = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7]
maxeval_ceiling = 2000

## Run the sweep

For each geoms state: update the resident fftPSFs once (`gather_PSFs_distributed`) and form
`noisy_image` once, then run a fresh reconstruction per `xtol_rel` value. Each value needs its own
real run -- unlike the earlier `maxeval`-sweep idea, whether/when `xtol_rel` actually triggers can
only be determined by letting NLopt evaluate it live; it isn't recoverable by truncating one longer
run after the fact.

In [ ]:
summary_rows = NamedTuple[]
object_opts = Dict{Tuple{String,Float64}, Any}()
noisy_images = Dict{String, Any}()

for (state_name, geoms_flat, α, β) in states
    geoms = End2EndThermalImg.unflatten_square_matrix(geoms_flat)
    End2EndThermalImg.gather_PSFs_distributed(geoms, freqs, php, imghp, PSF_scale, driver_plans_PSF, PSF_zcoords)
    noisy_image = End2EndThermalImg.make_noisy_image_from_3D_distributed(object, weights, noise, imghp.noise_type, php, imghp)
    noisy_images[state_name] = noisy_image

    for tol in xtol_rels
        rechp_test = with_xtol_rel(rechp, tol; maxeval = maxeval_ceiling)
        jhp_test = JobHyperParams(jhp; rechp = rechp_test)
        result = nothing
        t = @elapsed result = End2EndThermalImg.reconstruct_Tmap_and_zmap_distributed(noisy_image, weights, α, β, jhp_test; objective_scale = :auto)
        L = End2EndThermalImg.L_reconstruction_error(object, result.object_opt)
        n_iterations = length(result.objective_history)
        push!(summary_rows, (; state = state_name, xtol_rel = tol, return_value = result.return_value,
                              n_iterations = n_iterations, L = L, time_s = t))
        object_opts[(state_name, tol)] = result.object_opt
        println("$state_name  xtol_rel=$tol  -> $(result.return_value) after $n_iterations iters, " *
                "L=$(round(L, sigdigits=4)), $(round(t, digits=1))s")
    end
end

df = DataFrame(summary_rows)

## Summary table

In [ ]:
df

## Reconstruction error and iteration count vs. `xtol_rel`

In [ ]:
fig, axes = subplots(1, 2; figsize=(11, 4.5))

for (state_name, _, _, _) in states
    sub = df[df.state .== state_name, :]
    axes[0].plot(sub.xtol_rel, sub.L, marker="o", label=state_name)
    axes[1].plot(sub.xtol_rel, sub.n_iterations, marker="o", label=state_name)
end

axes[0].set_xscale("log")
axes[0].set_xlabel("xtol_rel")
axes[0].set_ylabel("L(object, object_opt)")
axes[0].set_title("Reconstruction error vs. xtol_rel")
axes[0].legend()
axes[0].grid(true, alpha=0.3)

axes[1].set_xscale("log")
axes[1].set_xlabel("xtol_rel")
axes[1].set_ylabel("Iterations used")
axes[1].axhline(maxeval_ceiling, color="gray", linestyle="--", label="maxeval ceiling")
axes[1].set_title("Iteration count vs. xtol_rel")
axes[1].legend()
axes[1].grid(true, alpha=0.3)

fig.tight_layout()
display(fig)

## Visual comparison: loosest vs. tightest tolerance

Reconstructed T/z maps and the formed-image/data-misfit row, for the loosest and tightest
`xtol_rel` tested, for each geoms state.

In [ ]:
tol_loose, tol_tight = maximum(xtol_rels), minimum(xtol_rels)

for (state_name, _, _, _) in states
    noisy_image = noisy_images[state_name]
    for tol in (tol_loose, tol_tight)
        object_opt = object_opts[(state_name, tol)]
        image_opt = End2EndThermalImg.make_image_from_3D_distributed(object_opt, weights, php, imghp)
        fig, ax = End2EndThermalImg.plot_reconstruction_results(object, object_opt, noisy_image, image_opt;
            extra_title = "$state_name | xtol_rel = $tol")
        display(fig)
    end
end